## Test Notebook
notebook for general tests of API functionality

In [ ]:
import os
import sys
from dotenv import load_dotenv
load_dotenv()

PROJECT_DIR = os.environ.get("PROJECT_DIR")
sys.path.insert(0,PROJECT_DIR)
DATA_DIR = os.environ.get("DATA_DIR")

### Testing Error Handling

In [ ]:
from app.schemas import LookupRequest

In [ ]:

from app.schemas import CardUrlError, CardFileTypeError
from pydantic import ValidationError

last_error = None
try:
    a = LookupRequest(
        # request_id="abc",
        request_id = "abc"*1000,
        source_system="dab",
        # card_url = "abcd"
        # card_url="https://www.google.com/"
        card_url="https://web.cs.wpi.edu/images/001.jpg",
    )
except Exception as error:
    last_error = error
    if isinstance(error,CardUrlError):
        print("is validation error")
    elif isinstance(error,CardFileTypeError):
        print("is type error")
    elif isinstance(error,ValidationError):
        print("is validation error")
    else:
        print(error)

In [ ]:
from fastapi import Request
from app.errors import app_error_handler,AppError

In [ ]:
request = Request({"type":"http","state":{"request_id":"joey_t"}})
exc = AppError("This is a test error")

In [ ]:
error = await app_error_handler(request,exc)

In [ ]:
error.body

In [ ]:
from fastapi import Depends

In [ ]:
from fastapi import FastAPI, Depends, Request, HTTPException
from fastapi.testclient import TestClient

app = FastAPI()
app.state.shared_thing = "I was created once at startup"

# ---------- BEFORE: the endpoint does its own setup ----------
@app.get("/before")
async def before(request: Request, token: str = ""):
    print("  endpoint: start")
    if token != "secret":                      # check written inline
        raise HTTPException(401, "bad token")
    thing = request.app.state.shared_thing     # fetch written inline
    print("  endpoint: body running")
    return {"thing": thing}

# ---------- AFTER: the setup moves into a dependency ----------
async def require_token(token: str = ""):
    print("  dependency: require_token runs")
    if token != "secret":
        raise HTTPException(401, "bad token")

async def get_thing(request: Request):
    print("  dependency: get_thing runs")
    return request.app.state.shared_thing

@app.get("/after", dependencies=[Depends(require_token)])
async def after(thing: str = Depends(get_thing)):
    print("  endpoint: body running (thing already in hand)")
    return {"thing": thing}

client = TestClient(app)

print("BEFORE, good token"); print(" ", client.get("/before?token=secret").json())
print("BEFORE, bad token");  print(" ", client.get("/before?token=nope").status_code)
print("AFTER, good token");  print(" ", client.get("/after?token=secret").json())
print("AFTER, bad token");   print(" ", client.get("/after?token=nope").status_code)

# Bonus: swap a dependency without touching the endpoint
app.dependency_overrides[get_thing] = lambda: "FAKE thing from override"
print("AFTER, overridden");  print(" ", client.get("/after?token=secret").json())

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

from app.errors import AppError, app_error_handler   # your real code

# a throwaway subclass, defined here only for the test
class TooBig(AppError):
    status_code = 413
    code = "too_big"

# throwaway app, so your real main.py isn't involved
test_app = FastAPI()
test_app.add_exception_handler(AppError, app_error_handler)   # the registration line under test

@test_app.get("/base")
async def raise_base():
    raise AppError("plain base error")

@test_app.get("/sub")
async def raise_sub():
    raise TooBig("image exceeds cap")

client = TestClient(test_app)

for path in ["/base", "/sub"]:
    r = client.get(path)
    print(path, r.status_code, r.json())

In [ ]:
test_app_2 = FastAPI()
from random import randint

@test_app_2.get("/test_endpoint")
async def return_some_data():
    return {"request_id":"ABC123","value":randint(100,1000)}

In [ ]:
client = TestClient(test_app_2)

client.get("/test_endpoint").json()

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from app.errors import AppError, VLMRateLimit, ResourceConnectionFailure, app_error_handler
from pydantic import BaseModel

class ErrorType(BaseModel):
    error_type: str

test_app = FastAPI()
test_app.add_exception_handler(AppError,app_error_handler)

@test_app.post("/error_test")
async def raise_error(value:ErrorType):
    if value.error_type == "vlm_rate_limit":
        raise VLMRateLimit("this is a vlm rate limit error")
    elif value.error_type == "resource_connection_failure":
        raise ResourceConnectionFailure("this is a resource connection failure")
    else:
        raise AppError("this is a generic error")
    
client = TestClient(test_app)

In [ ]:
import json
data = {"error_type":"resource_connection_failure"}
# data = json.loads('{"error_type":"vlm_rate_limit"}')

In [ ]:
response = client.post("/error_test",json = data)
print("Response Status Code:", response.status_code)
print("Response Text:", response.text)
print("Response Content:",response.content)

In [ ]:
from app.routers.routers import router

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
test_app = FastAPI()
test_app.include_router(router)
client = TestClient(test_app)

In [ ]:
client.get("/v1/health").text

### Testing the Happy Path

In [ ]:
from app.services.extractor import run_vlm_extraction_with_fake_extractor
from app.services.fetcher import run_image_byte_fetch
from app.settings import get_settings

In [ ]:
settings = get_settings()

In [ ]:
from httpx2 import AsyncClient
from anthropic import AsyncAnthropic
import asyncio

anthropic_client = AsyncAnthropic(api_key=settings.CLAUDE_API_KEY,timeout=settings.CLAUDE_TIMEOUT_SECONDS)
sem = asyncio.Semaphore(value = settings.CLAUDE_MAX_CONCURRENT_CALLS)
httpx2_client = AsyncClient(timeout=settings.FETCH_TIMEOUT_SECONDS)


request_id, buf, image_type, download_time = await run_image_byte_fetch(
    httpx2_client=httpx2_client,
    request_id="test_123",
    image_url = "https://web.cs.wpi.edu/~claypool/mmsys-dataset/2011/stanford/mvs_images/business_cards/Reference/001.jpg",
    max_bytes=settings.MAX_IMAGE_BYTES,
    allowed_image_hosts=settings.ALLOWED_IMAGE_HOSTS,
    allowed_image_types=settings.ALLOWED_IMAGE_TYPES,
    max_attempts=2,
)


await run_vlm_extraction_with_fake_extractor(
    anthropic_client=anthropic_client,
    sem=sem,
    request_id=request_id,
    image_bytes=buf,
    image_type=image_type,
    download_time = download_time,
    vlm_model=settings.CLAUDE_MODEL,
    max_attempts=2,
    fake_extractor=True
)